# 🍷 Wine Quality Prediction — Complete ML Project
**Dataset:** Red Wine Quality | **Task:** Binary Classification (GOOD vs BAD)

---

## STEP 1: Import Libraries

In [ ]:
# Core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ML tools
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

import warnings
warnings.filterwarnings('ignore')

print('All libraries imported successfully!')

## STEP 2: Load Dataset & Basic EDA

In [ ]:
# Upload winequality.csv to Colab first, then run this
df = pd.read_csv('winequality.csv')

print('=== SHAPE ===')
print(df.shape)

print('\n=== FIRST 5 ROWS (head) ===')
df.head()

In [ ]:
print('=== DATASET INFO (info) ===')
df.info()

In [ ]:
print('=== STATISTICAL SUMMARY (describe) ===')
df.describe()

## STEP 3: Check Missing Values

In [ ]:
print('=== MISSING VALUES ===')
print(df.isnull().sum())
print(f'\nTotal missing values: {df.isnull().sum().sum()}')

## STEP 4: Quality Distribution & Correlation Analysis

In [ ]:
# Quality distribution
print('=== QUALITY VALUE COUNTS ===')
print(df['quality'].value_counts().sort_index())

# Plot quality distribution
plt.figure(figsize=(8, 4))
df['quality'].value_counts().sort_index().plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Wine Quality Distribution')
plt.xlabel('Quality Score')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(12, 8))
correlation = df.corr()
sns.heatmap(correlation, annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

# Correlation with quality column specifically
print('\n=== CORRELATION WITH QUALITY ===')
print(df.corr()['quality'].sort_values(ascending=False))

## STEP 5: Create Binary Target — quality_label

In [ ]:
# Create binary label: 1 = GOOD (quality >= 7), 0 = BAD (quality < 7)
df['quality_label'] = df['quality'].apply(lambda x: 1 if x >= 7 else 0)

print('=== BINARY LABEL DISTRIBUTION ===')
print(df['quality_label'].value_counts())
print(f'\nGOOD wines (1): {df["quality_label"].sum()}')
print(f'BAD wines (0):  {len(df) - df["quality_label"].sum()}')

# Visualize
plt.figure(figsize=(6, 4))
df['quality_label'].value_counts().plot(kind='bar', color=['salmon', 'steelblue'], edgecolor='black')
plt.xticks([0, 1], ['GOOD (1)', 'BAD (0)'], rotation=0)
plt.title('Binary Quality Label Distribution')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

## STEP 6: Feature-Target Split & Train-Test Split

In [ ]:
# X = all features EXCEPT original quality & our new label
X = df.drop(columns=['quality', 'quality_label'])
y = df['quality_label']

print('Feature matrix shape (X):', X.shape)
print('Target vector shape (y): ', y.shape)
print('\nFeatures used:', list(X.columns))

In [ ]:
# 80% train, 20% test — random_state=42 for reproducibility
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print('Training set size:  ', X_train.shape[0])
print('Testing set size:   ', X_test.shape[0])
print('Train class balance:', y_train.value_counts().to_dict())
print('Test class balance: ', y_test.value_counts().to_dict())

## STEP 7: Logistic Regression — WITHOUT Scaling

In [ ]:
# Helper function to print all evaluation metrics
def evaluate_model(y_test, y_pred, model_name='Model'):
    print(f'\n========== {model_name} ==========')
    print(f'Accuracy : {accuracy_score(y_test, y_pred):.4f}')
    print(f'Precision: {precision_score(y_test, y_pred):.4f}')
    print(f'Recall   : {recall_score(y_test, y_pred):.4f}')
    print(f'F1-Score : {f1_score(y_test, y_pred):.4f}')
    print('\nClassification Report:')
    print(classification_report(y_test, y_pred, target_names=['BAD (0)', 'GOOD (1)']))
    print('Confusion Matrix:')
    cm = confusion_matrix(y_test, y_pred)
    print(cm)
    
    # Plot confusion matrix
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['BAD', 'GOOD'],
                yticklabels=['BAD', 'GOOD'])
    plt.title(f'Confusion Matrix — {model_name}')
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.tight_layout()
    plt.show()
    return accuracy_score(y_test, y_pred)

In [ ]:
# Logistic Regression WITHOUT scaling
lr_no_scale = LogisticRegression(random_state=42, max_iter=1000)
lr_no_scale.fit(X_train, y_train)
y_pred_lr_no = lr_no_scale.predict(X_test)

acc_no_scale = evaluate_model(y_test, y_pred_lr_no, 'Logistic Regression (No Scaling)')

## STEP 8: Apply StandardScaler & Logistic Regression WITH Scaling

In [ ]:
# StandardScaler: transforms data to mean=0, std=1
scaler = StandardScaler()

# IMPORTANT: fit only on training data, then transform both
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print('Before scaling — mean of first feature:', X_train.iloc[:, 0].mean().round(4))
print('After scaling  — mean of first feature:', X_train_scaled[:, 0].mean().round(4))
print('After scaling  — std of first feature: ', X_train_scaled[:, 0].std().round(4))

In [ ]:
# Logistic Regression WITH scaling
lr_scaled = LogisticRegression(random_state=42, max_iter=1000)
lr_scaled.fit(X_train_scaled, y_train)
y_pred_lr_sc = lr_scaled.predict(X_test_scaled)

acc_scaled = evaluate_model(y_test, y_pred_lr_sc, 'Logistic Regression (With Scaling)')

In [ ]:
# Compare before vs after scaling
print('=== SCALING COMPARISON ===')
print(f'Without Scaling — Accuracy: {acc_no_scale:.4f}')
print(f'With Scaling    — Accuracy: {acc_scaled:.4f}')
improvement = (acc_scaled - acc_no_scale) * 100
print(f'Improvement: {improvement:+.2f}%')

## STEP 9: Compare 3 Models — LR, KNN, Decision Tree

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'KNN (k=5)':           KNeighborsClassifier(n_neighbors=5),
    'Decision Tree':       DecisionTreeClassifier(random_state=42)
}

results = {}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    results[name] = {
        'Accuracy':  round(accuracy_score(y_test, y_pred), 4),
        'Precision': round(precision_score(y_test, y_pred), 4),
        'Recall':    round(recall_score(y_test, y_pred), 4),
        'F1-Score':  round(f1_score(y_test, y_pred), 4)
    }

results_df = pd.DataFrame(results).T
print('=== MODEL COMPARISON TABLE ===')
print(results_df)

best_model_name = results_df['F1-Score'].idxmax()
print(f'\n>>> BEST MODEL by F1-Score: {best_model_name} ({results_df.loc[best_model_name, "F1-Score"]})')

In [ ]:
# Bar chart comparison
results_df.plot(kind='bar', figsize=(10, 5), colormap='viridis', edgecolor='black')
plt.title('Model Comparison — All Metrics')
plt.xticks(rotation=15)
plt.ylim(0.5, 1.05)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

## STEP 10: Hyperparameter Tuning with GridSearchCV

In [ ]:
# Tuning Decision Tree (commonly the best on this dataset)
# Adjust model name if your best model was different

param_grid = {
    'max_depth':        [3, 5, 7, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf':  [1, 2, 4],
    'criterion':         ['gini', 'entropy']
}

grid_search = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train_scaled, y_train)

print('Best Parameters:', grid_search.best_params_)
print('Best CV F1-Score:', round(grid_search.best_score_, 4))

In [ ]:
# Evaluate the tuned model
best_model = grid_search.best_estimator_
y_pred_tuned = best_model.predict(X_test_scaled)

evaluate_model(y_test, y_pred_tuned, 'Decision Tree (After GridSearchCV Tuning)')

## STEP 11: Feature Importance Analysis

In [ ]:
# Feature importance from Decision Tree
feature_importance = pd.Series(
    best_model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

print('=== FEATURE IMPORTANCE (Decision Tree) ===')
for feat, imp in feature_importance.items():
    bar = '█' * int(imp * 50)
    print(f'{feat:25s}: {imp:.4f}  {bar}')

# Bar chart
plt.figure(figsize=(10, 5))
feature_importance.plot(kind='barh', color='steelblue', edgecolor='black')
plt.title('Feature Importance — Tuned Decision Tree')
plt.xlabel('Importance Score')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Bonus: Logistic Regression coefficients as feature importance
lr_coef = pd.Series(
    np.abs(lr_scaled.coef_[0]),
    index=X.columns
).sort_values(ascending=False)

print('=== FEATURE IMPORTANCE (Logistic Regression — absolute coefficients) ===')
print(lr_coef.round(4))

## STEP 12: Final Summary

In [ ]:
print('=' * 55)
print('       WINE QUALITY ML PROJECT — FINAL SUMMARY')
print('=' * 55)
print(f'Dataset: {df.shape[0]} rows, {X.shape[1]} features')
print(f'Target: Binary — GOOD (1) vs BAD (0)')
print(f'GOOD wines: {y.sum()} | BAD wines: {len(y)-y.sum()}')
print()
print('--- Model Results (on scaled data) ---')
print(results_df.to_string())
print()
print(f'Best model: {best_model_name}')
print(f'After GridSearchCV tuning:')
print(f'  F1-Score  = {f1_score(y_test, y_pred_tuned):.4f}')
print(f'  Accuracy  = {accuracy_score(y_test, y_pred_tuned):.4f}')
print()
print('Top 3 Important Features:')
for i, (feat, val) in enumerate(feature_importance.head(3).items(), 1):
    print(f'  {i}. {feat} ({val:.4f})')
print('=' * 55)